In [117]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
import openpyxl
import itertools
from IPython.display import display

Now, we create plots for every country and every first order region with daily observation counts and running monthly means.

In [137]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = pd.to_datetime(df["date"])

start_date = pd.to_datetime("2017-02-01")
end_date   = pd.to_datetime("2026-06-30")

def plot_country(country, split_by_year=True):
    base_path = f"../Products/Timelines_by_Country/{country}"
    os.makedirs(base_path, exist_ok=True)

    df_filtered = df[df["Country"] == country].copy()
    _plot(df_filtered, base_path, title=country, filename_prefix=country, split_by_year=split_by_year)

def plot_region(country, region, split_by_year=True):
    
    base_path = f"../Products/Timelines_by_Country/{country}/regions/{region}"
    os.makedirs(base_path, exist_ok=True)

    df_filtered = df[(df["Country"] == country) & (df["Region"] == region)].copy()
    _plot(df_filtered, base_path, title=f"{country} - {region}", filename_prefix=f"{country}_{region}", split_by_year=split_by_year)

def _plot(df_filtered, base_path, title, filename_prefix, split_by_year=True):
    # internal helping function
    
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")

    daily = (
        df_filtered
        .groupby("date")
        .size()
        .rename("daily_observations")
        .reindex(full_range, fill_value=0)
    )
    rolling_30 = daily.rolling(window=30, min_periods=1, center=True).mean()
    data = pd.DataFrame({"daily": daily, "rolling_30": rolling_30})

    # Full plot
    _save_plot(data.index, data["daily"], data["rolling_30"], 
               title=title, 
               path=f"{base_path}/timeline_{filename_prefix}_full.png")

    if split_by_year:
        for year in range(2017, 2027):
            start = pd.to_datetime(f"{year}-01-01")
            end   = pd.to_datetime(f"{year}-12-31")
            if year == 2017: start = pd.to_datetime("2017-02-01")
            if year == 2026: end = pd.to_datetime("2026-06-30")

            df_year = data.loc[start:end]
            if df_year["daily"].sum() == 0:
                continue

            _save_plot(df_year.index, df_year["daily"], df_year["rolling_30"],
                       title=f"{title} - {year}",
                       path=f"{base_path}/timeline_{filename_prefix}_{year}.png")

def _save_plot(index, daily, rolling, title, path):
    
    plt.figure(figsize=(16, 8))
    plt.plot(index, daily)
    plt.plot(index, rolling)
    plt.title(title, fontsize=15)
    plt.xlabel("Date")
    plt.ylabel("Observations")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.6)
    plt.legend(["Daily observations", "Running 30-day mean"], loc="upper right")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_32940\789466163.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

In [74]:
for country in df["Country"].unique():
    plot_country(country)

for (country, region) in df[["Country", "Region"]].drop_duplicates().values:
    plot_region(country, region)

Make dataframes of daily observations in each and every first order subregion and in every country of the world, in order to later detect events in the data (=unusual patterns with spikes in observations).

In [48]:
df = pd.read_csv("../CWData_clean7.csv")
df["date"] = pd.to_datetime(df["date"])

daily_region = (
    df
    .groupby(["Country", "Region", "date"])
    .agg(
        daily_obs=("date", "size"),
        contributors=("created_by", set)
    )
    .reset_index()
)

daily_country = (
    df
    .groupby(["Country", "date"])
    .agg(
        daily_obs=("date", "size"),
        contributors=("created_by", set)
    )
    .reset_index()
)

daily_global = (
    df
    .groupby("date")
    .agg(
        daily_obs=("date", "size"),
        contributors=("created_by", set)
    )
    .reset_index()
)

C:\Users\yanni\AppData\Local\Temp\ipykernel_32940\2003978235.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

calculate z-score (using daily observations, 60-day mean observation counts and 60-day standard deviation) for every day for every region and every country.

In [49]:
# uses daily_country, daily_region and daily_global

full_range = pd.date_range("2017-02-01", "2026-06-30", freq="D")

all_regions = []

lag = 0
for (country, region), group in daily_region.groupby(["Country", "Region"]):

    d = group.set_index("date").sort_index()
    d = d.reindex(full_range)
    d["daily_obs"] = d["daily_obs"].fillna(0)
    d["contributors"] = d["contributors"].apply(lambda x: x if isinstance(x, set) else set())
    d["Country"] = country
    d["Region"] = region
    d["mean_60"] = d["daily_obs"].shift(lag).rolling(60).mean()
    d["std_60"]  = d["daily_obs"].shift(lag).rolling(60).std()
    d["z_score"] = (d["daily_obs"] - d["mean_60"]) / d["std_60"]

    d = d.reset_index().rename(columns={"index": "date"})

    all_regions.append(d)

region_events_df = pd.concat(all_regions, ignore_index=True)

all_countries = []

for country, group in daily_country.groupby("Country"):

    d = group.set_index("date").sort_index()
    d = d.reindex(full_range)
    d["daily_obs"] = d["daily_obs"].fillna(0)
    d["contributors"] = d["contributors"].apply(lambda x: x if isinstance(x, set) else set())
    d["Country"] = country
    d["mean_60"] = d["daily_obs"].shift(lag).rolling(60).mean()
    d["std_60"]  = d["daily_obs"].shift(lag).rolling(60).std()
    d["z_score"] = (d["daily_obs"] - d["mean_60"]) / d["std_60"]
    
    d = d.reset_index().rename(columns={"index": "date"})
    
    all_countries.append(d)

country_events_df = pd.concat(all_countries, ignore_index=True)

d = daily_global.set_index("date").sort_index().reindex(full_range)
d["daily_obs"] = d["daily_obs"].fillna(0)
d["contributors"] = d["contributors"].apply(lambda x: x if isinstance(x, set) else set())
d["mean_60"] = d["daily_obs"].shift(lag).rolling(60).mean()
d["std_60"]  = d["daily_obs"].shift(lag).rolling(60).std()
d["z_score"] = (d["daily_obs"] - d["mean_60"]) / d["std_60"]
d = d.reset_index().rename(columns={"index": "date"})
d["Country"] = "Global"

global_events_df = d

Function that detects events

In [50]:
def detect_events(df_input, group_col, z_threshold=2, block_length=3, min_contributors=2): # everything above threshold counts as unusual and if more than x days are above it, it's an event
    
    df_input = df_input.copy().sort_values("date")
    df_input["is_event_day"] = (df_input["z_score"] > z_threshold) & (df_input["daily_obs"] >= 2) # day is event if z-score above threshold AND if it has more than 1 obs

    if group_col == "Global":
        df_input["rolling_sum"] = df_input["is_event_day"].rolling(block_length).sum()
        df_input["event_block"] = pd.concat(
            [df_input["rolling_sum"].shift(-i) for i in range(block_length)], axis=1
        ).max(axis=1) >= block_length

    else:
        df_input["rolling_sum"] = (
            df_input
            .groupby(group_col)["is_event_day"]
            .transform(lambda x: x.rolling(block_length).sum())
        )

        df_input["event_block"] = (
            df_input
            .groupby(group_col)["rolling_sum"]
            .transform(lambda x: pd.concat(
                [x.shift(-i) for i in range(block_length)], axis=1
            ).max(axis=1) >= block_length)
        )

    periods = []

    if group_col == "Global":
        group_keys = None
    elif group_col == "Region":
        group_keys = ["Country", "Region"]
    else:
        group_keys = ["Country"]

    if group_keys is None:
        groups = [(("Global",), df_input)]
    else:
        groups = df_input.groupby(group_keys)

    for key, d in groups:

        d = d.copy().sort_values("date")

        if group_col == "Global":
            country = "Global"
            region = None
        elif group_col == "Region":
            country, region = key
        else:
            country = key[0]
            region = None

        d["event_shift"] = d["event_block"].shift(1, fill_value=False)

        starts = d[(d["event_block"] == True) & (d["event_shift"] == False)]
        ends = d[(d["event_block"] == False) & (d["event_shift"] == True)]

        start_dates = starts["date"].values
        end_dates   = ends["date"].values

        if len(start_dates) > len(end_dates):
            end_dates = list(end_dates) + [d["date"].iloc[-1]]

        for s, e in zip(start_dates, end_dates):
            e_adj = pd.to_datetime(e) - pd.Timedelta(days=1)
            mask = (d["date"] >= s) & (d["date"] <= e_adj)
            z_vals = d.loc[mask, "z_score"]

            contributor_sets = d.loc[mask, "contributors"]
            all_contributors = set().union(*contributor_sets) if len(contributor_sets) > 0 else set()
            n_observers = len(all_contributors)
            if n_observers < min_contributors:
                continue

            periods.append({
                "Country":    country,
                "Region":     region if group_col == "Region" else None,
                "level":      group_col if group_col != "Global" else "Global",
                "start_date": s,
                "end_date":   pd.to_datetime(e) - pd.Timedelta(days=1),
                "event_length (days)": (pd.to_datetime(e) - pd.to_datetime(s)).days,
                "max_z_score": z_vals.max(),
                "mean_z_score": z_vals.mean(),
                "n_observers": n_observers
            })

    return pd.DataFrame(periods)

CrowdWater outreach events

In [141]:
outreach = pd.read_excel("../CW_Outreaches/CW_outreach_merged.xlsx", sheet_name="Tabelle1")
outreach["Date"] = pd.to_datetime(outreach["Date"])

outreach.head()

,Date,Category,City,Region,Country,Details
0,2020-01-16,Non-Swiss,Munich,Bayern,Germany,Presentation TU München
1,2020-02-19,Swiss,NaN,NaN,Switzerland,Brown Bag Lunch CitSci Center
2,2020-02-27,Swiss,NaN,NaN,Switzerland,Presentation for Mercator Price
3,2020-03-05,International,NaN,NaN,NaN,Advertisement CrowdWater Game
4,2020-03-06,International,NaN,NaN,NaN,Newsletter: Spring 2020: Goodbye Barbara and Simon


In [142]:
outreach_no_international = outreach[(outreach["Category"] != "International")]

outreach_no_international.head()

,Date,Category,City,Region,Country,Details
0,2020-01-16,Non-Swiss,Munich,Bayern,Germany,Presentation TU München
1,2020-02-19,Swiss,NaN,NaN,Switzerland,Brown Bag Lunch CitSci Center
2,2020-02-27,Swiss,NaN,NaN,Switzerland,Presentation for Mercator Price
5,2020-03-18,Swiss,NaN,NaN,Switzerland,Defense Simon and Barbara
6,2020-04-27,Non-Swiss,Salzburg,Salzburg,Austria,Presentation at Universität Salzburg


In [143]:
outreach_only_non_swiss = outreach[(outreach["Category"] != "International") & (outreach["Category"] != "Swiss")]

outreach_only_non_swiss.head()

,Date,Category,City,Region,Country,Details
0,2020-01-16,Non-Swiss,Munich,Bayern,Germany,Presentation TU München
6,2020-04-27,Non-Swiss,Salzburg,Salzburg,Austria,Presentation at Universität Salzburg
14,2020-12-01,Non-Swiss,Salzburg,Salzburg,Austria,Presentation at Universität Salzburg
25,2021-05-07,Non-Swiss,Havana,Illinois,United States of America,Presentation at Society for Freshwater Science
26,2021-05-18,Non-Swiss,Birmingham,Birmingham,United Kingdom,Presentation at University of Birmingham


Multiple different combinations of block length, minimum number of contributors and z-score threshold

In [144]:
def evaluate_against_outreach(events_table, outreach_df, tolerance_days=10):
    """
    Matching logic:
    - If Region is specified: match against Region-level events (Country + Region)
    - If only Country is specified: match against Country-level events
    - If Category == "International" (no specific location): match against Global-level events
    """
    events_table = events_table.copy()
    events_table["start_date"] = pd.to_datetime(events_table["start_date"])
    events_table["end_date"] = pd.to_datetime(events_table["end_date"])

    outreach_df = outreach_df.copy()
    outreach_df["Date"] = pd.to_datetime(outreach_df["Date"])

    outreach_min_date = pd.to_datetime("2020-01-01") - pd.Timedelta(days=5) # because official CrowdWater outreach data is only available from 2020 onwards
    events_table = events_table[events_table["end_date"] >= outreach_min_date].copy()

    matched_event_indices = set()
    match_results = []

    for idx, event in outreach_df.iterrows():
        event_date = event["Date"]
        country = event.get("Country", None)
        region = event.get("Region", None)
        category = event.get("Category", None)

        window_start = event_date - pd.Timedelta(days=5)
        window_end = event_date + pd.Timedelta(days=tolerance_days)

        if category == "International":
            candidates = events_table[
                (events_table["level"] == "Global") &
                (events_table["start_date"] <= window_end) &
                (events_table["end_date"] >= window_start)
            ]
            match_level = "Global"

        elif pd.notna(region):
            # Region-Outreach: against Region-Level (e.g. the canton) AND Country-Level (e.g., whole of Switzerland)
            candidates = events_table[
                (
                    ((events_table["level"] == "Region") & (events_table["Region"] == region)) |
                    (events_table["level"] == "Country")
                ) &
                (events_table["Country"] == country) &
                (events_table["start_date"] <= window_end) &
                (events_table["end_date"] >= window_start)
            ]
            match_level = "Region+Country"

        elif pd.notna(country):
            # Country-Outreach: against Country-Level AND all Region-Level of this country
            candidates = events_table[
                (events_table["level"].isin(["Country", "Region"])) &
                (events_table["Country"] == country) &
                (events_table["start_date"] <= window_end) &
                (events_table["end_date"] >= window_start)
            ]
            match_level = "Country+Region"

        else:
            match_results.append({
                "outreach_idx": idx, "date": event_date, "category": category,
                "country": country, "region": region,
                "matched": False, "match_level": None,
                "reason": "no location info and not International"
            })
            continue

        matched = len(candidates) > 0
        if matched:
            matched_event_indices.update(candidates.index)

        match_results.append({
            "outreach_idx": idx,
            "date": event_date,
            "category": category,
            "country": country,
            "region": region,
            "matched": matched,
            "match_level": match_level,
            "n_candidates": len(candidates)
        })

    match_details = pd.DataFrame(match_results)

    n_valid_outreach = (~match_details["match_level"].isna()).sum() if "match_level" in match_details else len(match_details)
    n_matched_outreach = match_details["matched"].sum()

    recall = n_matched_outreach / n_valid_outreach if n_valid_outreach > 0 else float("nan")
    precision = len(matched_event_indices) / len(events_table) if len(events_table) > 0 else float("nan")

    summary = {
        "recall": recall,
        "precision": precision,
        "n_events_detected": len(events_table),
        "n_events_matched": len(matched_event_indices),
        "n_outreach_valid": n_valid_outreach,
        "n_outreach_matched": n_matched_outreach,
    }

    return summary, match_details

In [145]:
def run_grid_search(country_events_df, region_events_df, global_events_df, outreach_df,
                     z_thresholds, block_lengths, min_contributors_list, tolerance_days=10):

    grid_results = []
    events_by_config = {}

    for z, bl, mc in itertools.product(z_thresholds, block_lengths, min_contributors_list):

        global_ev  = detect_events(global_events_df,  "Global",  z_threshold=z, block_length=bl, min_contributors=mc)
        country_ev = detect_events(country_events_df, "Country", z_threshold=z, block_length=bl, min_contributors=mc)
        region_ev  = detect_events(region_events_df,  "Region",  z_threshold=z, block_length=bl, min_contributors=mc)

        combined_events = pd.concat([global_ev, country_ev, region_ev], ignore_index=True)

        summary, details = evaluate_against_outreach(combined_events, outreach_df, tolerance_days=tolerance_days)

        config_key = (z, bl, mc)
        events_by_config[config_key] = {
            "events": combined_events,
            "match_details": details
        }

        grid_results.append({
            "z_threshold": z,
            "block_length": bl,
            "min_contributors": mc,
            **summary
        })

    grid_df = pd.DataFrame(grid_results)
    grid_df["f1"] = 2 * (grid_df["precision"] * grid_df["recall"]) / (grid_df["precision"] + grid_df["recall"])

    return grid_df, events_by_config

In [147]:
grid_df_all, events_by_config_all = run_grid_search(
    country_events_df, region_events_df, global_events_df, outreach,
    z_thresholds=[1.5, 2, 2.5, 3],
    block_lengths=[2, 3, 4],
    min_contributors_list=[1, 2, 3],
    tolerance_days=7
)

pd.set_option("display.max_rows", None)
grid_df_all.sort_values("f1", ascending=False)

,z_threshold,block_length,min_contributors,recall,precision,n_events_detected,n_events_matched,n_outreach_valid,n_outreach_matched,f1
2,1.5,2,3,0.212766,0.184615,260,48,141,30,0.197694
1,1.5,2,2,0.248227,0.152882,399,61,141,35,0.189223
10,2.0,2,2,0.205674,0.158491,265,42,141,29,0.179026
11,2.0,2,3,0.170213,0.182927,164,30,141,24,0.176341
4,1.5,3,2,0.134752,0.247191,89,22,141,19,0.174421
19,2.5,2,2,0.177305,0.164835,182,30,141,25,0.170843
0,1.5,2,1,0.297872,0.115097,669,77,141,42,0.166038
5,1.5,3,3,0.120567,0.250000,72,18,141,17,0.162679
3,1.5,3,1,0.148936,0.175182,137,24,141,21,0.160997
9,2.0,2,1,0.248227,0.116129,465,54,141,35,0.158232


In [148]:
grid_df_no_international, events_by_config_no_international = run_grid_search(
    country_events_df, region_events_df, global_events_df, outreach_no_international,
    z_thresholds=[1.5, 2, 2.5, 3],
    block_lengths=[2, 3, 4],
    min_contributors_list=[1, 2, 3],
    tolerance_days=7
)

pd.set_option("display.max_rows", None)
grid_df_no_international.sort_values("f1", ascending=False)

,z_threshold,block_length,min_contributors,recall,precision,n_events_detected,n_events_matched,n_outreach_valid,n_outreach_matched,f1
4,1.5,3,2,0.188119,0.247191,89,22,101,19,0.213647
2,1.5,2,3,0.267327,0.173077,260,45,101,27,0.210117
5,1.5,3,3,0.168317,0.250000,72,18,101,17,0.201183
1,1.5,2,2,0.316832,0.145363,399,58,101,32,0.199291
11,2.0,2,3,0.227723,0.176829,164,29,101,23,0.199075
10,2.0,2,2,0.277228,0.154717,265,41,101,28,0.198599
19,2.5,2,2,0.247525,0.164835,182,30,101,25,0.197889
3,1.5,3,1,0.207921,0.175182,137,24,101,21,0.190153
13,2.0,3,2,0.128713,0.285714,49,14,101,13,0.177474
20,2.5,2,3,0.168317,0.181818,110,20,101,17,0.174807


In [149]:
grid_df_only_non_swiss, events_by_config_only_non_swiss = run_grid_search(
    country_events_df, region_events_df, global_events_df, outreach_only_non_swiss,
    z_thresholds=[1.5, 2, 2.5, 3],
    block_lengths=[2, 3, 4],
    min_contributors_list=[1, 2, 3],
    tolerance_days=7
)

pd.set_option("display.max_rows", None)
grid_df_only_non_swiss.sort_values("f1", ascending=False)

,z_threshold,block_length,min_contributors,recall,precision,n_events_detected,n_events_matched,n_outreach_valid,n_outreach_matched,f1
5,1.5,3,3,0.108108,0.069444,72,5,37,4,0.084567
14,2.0,3,3,0.081081,0.085714,35,3,37,3,0.083333
4,1.5,3,2,0.108108,0.056180,89,5,37,4,0.073937
13,2.0,3,2,0.081081,0.061224,49,3,37,3,0.069767
8,1.5,4,3,0.054054,0.068966,29,2,37,2,0.060606
12,2.0,3,1,0.081081,0.041667,72,3,37,3,0.055046
3,1.5,3,1,0.108108,0.036496,137,5,37,4,0.054570
7,1.5,4,2,0.054054,0.052632,38,2,37,2,0.053333
2,1.5,2,3,0.108108,0.034615,260,9,37,4,0.052440
11,2.0,2,3,0.081081,0.036585,164,6,37,3,0.050420


Which events are frequently detected?

In [150]:
def summarize_frequent_detections(events_by_config, top_n=15): # which anomaly get detected frequently (does not have to be an outreach event)
    all_events = []
    for config_key, data in events_by_config.items():
        z, bl, mc = config_key
        ev = data["events"].copy()
        ev["z_threshold"] = z
        ev["block_length"] = bl
        ev["min_contributors"] = mc
        all_events.append(ev)

    combined = pd.concat(all_events, ignore_index=True)

    # frequency per location
    location_counts = (
        combined
        .groupby(["level", "Country", "Region"], dropna=False)
        .size()
        .rename("n_detections_across_configs")
        .reset_index()
        .sort_values("n_detections_across_configs", ascending=False)
    )

    return location_counts.head(top_n)


def summarize_frequent_outreach_matches(events_by_config, top_n=15): # which outreach event gets detected most frequently
    all_matches = []
    for config_key, data in events_by_config.items():
        z, bl, mc = config_key
        details = data["match_details"].copy()
        details["z_threshold"] = z
        details["block_length"] = bl
        details["min_contributors"] = mc
        all_matches.append(details)

    combined = pd.concat(all_matches, ignore_index=True)

    match_counts = (
        combined[combined["matched"] == True]
        .groupby(["outreach_idx", "date", "category", "country", "region"], dropna=False)
        .size()
        .rename("n_configs_matched")
        .reset_index()
        .sort_values("n_configs_matched", ascending=False)
    )

    return match_counts.head(top_n)

In [151]:
print("Top detected locations (all outreaches):")
display(summarize_frequent_detections(events_by_config_all))

print("\nmost frequently matched outreach events (all outreaches):")
display(summarize_frequent_outreach_matches(events_by_config_all))

print("\nmost frequently matched outreach events (no international outreaches):")
display(summarize_frequent_outreach_matches(events_by_config_no_international))

print("\nmost frequently matched outreach events (only non-swiss outreaches):")
display(summarize_frequent_outreach_matches(events_by_config_only_non_swiss))

Top detected locations (all outreaches):


,level,Country,Region,n_detections_across_configs
40,Global,Global,NaN,282
126,Region,Switzerland,Aargau,249
37,Country,Switzerland,NaN,234
138,Region,Switzerland,Zürich,225
131,Region,Switzerland,Sankt Gallen,165
21,Country,Kyrgyzstan,NaN,152
14,Country,Germany,NaN,140
39,Country,United States of America,NaN,136
77,Region,Germany,Baden-Württemberg,105
101,Region,Kyrgyzstan,Jalal-Abad,104



most frequently matched outreach events (all outreaches):


,outreach_idx,date,category,country,region,n_configs_matched
41,141,2026-06-18,Swiss,Switzerland,NaN,36
4,10,2020-09-15,Swiss,Switzerland,NaN,27
2,8,2020-05-22,Swiss,Switzerland,NaN,27
20,69,2022-10-28,Swiss,Switzerland,NaN,24
19,67,2022-10-12,Swiss,Switzerland,NaN,24
22,76,2023-02-08,Swiss,Switzerland,NaN,23
23,77,2023-02-09,Swiss,Switzerland,NaN,23
24,78,2023-02-10,Swiss,Switzerland,NaN,23
30,101,2024-03-24,Non-Swiss,Germany,NaN,21
25,80,2023-03-15,Non-Swiss,Costa Rica,NaN,18



most frequently matched outreach events (no international outreaches):


,outreach_idx,date,category,country,region,n_configs_matched
38,141,2026-06-18,Swiss,Switzerland,NaN,36
3,10,2020-09-15,Swiss,Switzerland,NaN,27
2,8,2020-05-22,Swiss,Switzerland,NaN,27
19,69,2022-10-28,Swiss,Switzerland,NaN,24
18,67,2022-10-12,Swiss,Switzerland,NaN,24
22,77,2023-02-09,Swiss,Switzerland,NaN,23
23,78,2023-02-10,Swiss,Switzerland,NaN,23
21,76,2023-02-08,Swiss,Switzerland,NaN,23
29,101,2024-03-24,Non-Swiss,Germany,NaN,21
24,80,2023-03-15,Non-Swiss,Costa Rica,NaN,18



most frequently matched outreach events (only non-swiss outreaches):


,outreach_idx,date,category,country,region,n_configs_matched
5,101,2024-03-24,Non-Swiss,Germany,NaN,21
2,80,2023-03-15,Non-Swiss,Costa Rica,NaN,18
4,100,2024-03-20,Non-Swiss,Germany,Berlin,18
1,49,2022-03-30,Non-Swiss,Germany,Nordrhein-Westfalen,8
3,81,2023-03-21,Non-Swiss,Germany,Berlin,6
0,27,2021-05-20,Non-Swiss,Ecuador,Azuay,4


In [ ]:
def inspect_event(df_raw, country_events_df, country, event_date, window=10):
    event_date = pd.to_datetime(event_date)

    
    mask = (country_events_df["Country"] == country) & \
           (country_events_df["date"] >= event_date - pd.Timedelta(days=window)) & \
           (country_events_df["date"] <= event_date + pd.Timedelta(days=window))
    result = country_events_df.loc[mask, ["date", "daily_obs", "z_score"]].sort_values("date").copy()

    
    df_country = df_raw[df_raw["Country"] == country]

    contributors_per_day = (
        df_country
        .groupby("date")["created_by"]
        .apply(lambda x: sorted(x.unique().tolist()))
    )

    result["contributors"] = result["date"].map(contributors_per_day)
    result["contributors"] = result["contributors"].apply(lambda x: x if isinstance(x, list) else [])
    result["n_contributors"] = result["contributors"].apply(len)

    return result

pd.set_option("display.max_colwidth", None)
inspect_event(df, country_events_df, "Chile", "2021-08-02")

,date,daily_obs,z_score,contributors,n_contributors
49751,2021-07-23,0.0,-0.662609,[],0
49752,2021-07-24,3.0,0.603991,[52028],1
49753,2021-07-25,4.0,1.000806,[52028],1
49754,2021-07-26,4.0,0.968708,[52028],1
49755,2021-07-27,7.0,2.111336,"[28674, 52028]",2
49756,2021-07-28,2.0,0.074532,[52028],1
49757,2021-07-29,4.0,0.868401,[52028],1
49758,2021-07-30,1.0,-0.351375,[52028],1
49759,2021-07-31,3.0,0.490791,[52028],1
49760,2021-08-01,0.0,-0.753468,[],0


Change-Point-Detection (C. Truong, L. Oudre, N. Vayatis. Selective review of offline change point detection methods. Signal Processing, 167:107299, 2020.)

In [9]:
def detect_change_points(df_input, group_col, pen, min_size):

    change_points_list = []

    for group_val in df_input[group_col].unique():

        d = df_input[df_input[group_col] == group_val].copy()
        
        # if region has less than five total observations, skip it
        if d["daily_obs"].max() < 5:
            continue
        country = d["Country"].iloc[0]

        # remove time before first activity in region
        #first_activity = d[d["daily_obs"] > 0]["date"]
        #if len(first_activity) == 0:
            #continue

        #d = d[d["date"] >= first_activity.min()]

        # weekly aggregation
        d_weekly = (
            d.set_index("date")
             .resample("W")["daily_obs"]
             .sum()
             .reset_index()
        )

        d_weekly["cum_obs"] = d_weekly["daily_obs"].cumsum()

        d_weekly["growth"] = d_weekly["daily_obs"].pct_change().fillna(0)

        signal = d_weekly["daily_obs"].values

        if np.sum(signal) == 0:
            continue

        # Model
        model = rpt.Pelt(model="l2", min_size=min_size).fit(signal)

        if len(signal) < 2 * min_size:
            continue

        # calculate breakpoints
        breakpoints = model.predict(pen=pen)

        # predict() returns indices
        # last point is always length (--> ignore that one)
        for bp in breakpoints[:-1]:

            change_points_list.append({
                "Country": country,
                "Region": group_val if group_col == "Region" else None,
                "level": group_col,
                "change_point_date": d_weekly["date"].iloc[bp]
            })

    return pd.DataFrame(change_points_list)

In [10]:
# uses country_events_df and region_events_df

cp_list = []
cp_list.append(detect_change_points(country_events_df, "Country", pen=80, min_size=8))
cp_list.append(detect_change_points(region_events_df, "Region", pen=80, min_size=8))

pd.set_option("display.max_rows", None)
cp_table = pd.concat(cp_list, ignore_index=True)
cp_table

,Country,Region,level,change_point_date
0,Australia,None,Country,2021-05-30
1,Australia,None,Country,2021-10-17
2,Australia,None,Country,2022-01-30
3,Austria,None,Country,2018-01-21
4,Austria,None,Country,2018-04-01
5,Austria,None,Country,2018-08-19
6,Austria,None,Country,2019-01-06
7,Austria,None,Country,2019-11-17
8,Austria,None,Country,2020-07-19
9,Austria,None,Country,2021-05-30


Plotting the daily median distance to Zurich of all observations over time, as well as annual plots to see it better

In [11]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = pd.to_datetime(df["date"])

zurich = (47.3769, 8.5417)

df["distance_zurich"] = df.apply(
    lambda row: haversine(
        zurich,
        (row["latitude"], row["longitude"]),
        unit="km"
    ),
    axis=1
)

daily_distance = (
    df
    .groupby("date")["distance_zurich"]
    .median()
    .reset_index()
)

daily_distance["distance_30d_mean"] = (
    daily_distance["distance_zurich"]
    .rolling(window=30, min_periods=1)
    .mean()
)

plt.figure(figsize=(16,8))

plt.plot(
    daily_distance["date"],
    daily_distance["distance_zurich"],
    alpha=0.6,
    label="Daily median"
)

plt.plot(
    daily_distance["date"],
    daily_distance["distance_30d_mean"],
    linewidth=2,
    label="30-day running mean"
)

plt.ylabel("Distance [km]", fontsize=12)
plt.xlabel("Date", fontsize=12)
plt.title("Daily Median Distance of Observations to Zurich over time", fontsize=15)
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.4)
plt.ylim(0,8000)
plt.savefig("../Products/Distance_to_Zurich/Median_Distance_to_Zurich_full.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_3488\372182383.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 42

annual distance-to-Zurich plots

In [12]:
# uses daily_distance

daily_distance["year"] = daily_distance["date"].dt.year
years = sorted(daily_distance["year"].unique())

for year in years:

    data = daily_distance[daily_distance["year"] == year]
    year_median = df[df["date"].dt.year == year]["distance_zurich"].median()

    plt.figure(figsize=(16,8))

    plt.plot(
        data["date"],
        data["distance_zurich"],
        alpha=0.6,
        label="Daily median"
    )

    plt.plot(
        data["date"],
        data["distance_30d_mean"],
        linewidth=2,
        label="30-day running mean"
    )

    plt.text(
        0.02,
        0.95,
        f"Yearly median distance: {year_median:.0f} km",
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top"
    )

    plt.ylabel("Distance [km]", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    plt.title(f"Daily Median Distance of Observations to Zurich over time - {year}", fontsize=15)
    plt.legend(loc="upper right")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.4)
    plt.ylim(0,8000)
    plt.savefig(f"../Products/Distance_to_Zurich/Median_Distance_to_Zurich_{year}.png", dpi=300, bbox_inches="tight")
    plt.close()